# What-If Analysis: Exploring Configuration Space

This notebook demonstrates sensitivity sweeps and hardware comparisons using llm-perf.

In [ ]:
from llm_perf.config import load_model_config, load_hardware_config, WorkloadConfig, ParallelismConfig
from llm_perf.model import LLMPerformanceModel

# Shared baseline
model = load_model_config("../configs/models/llama3_1_8b.yaml")
hw_910c = load_hardware_config("../configs/hardware/ascend_910c.yaml")
hw_cm = load_hardware_config("../configs/hardware/cloudmatrix_384.yaml")

base_rl = WorkloadConfig(
    total_prompts=10000,
    group_size=8,
    avg_prompt_len=512,
    avg_response_len=2048,
    max_response_len=4096,
    gen_batch_size=64,
)
gen_p = ParallelismConfig(tp=8, dp=8)
train_p = ParallelismConfig(tp=8, dp=8)

## Response Length Impact

In [ ]:
perf = LLMPerformanceModel(model, hw_910c)

print(f"{'Avg Response Len':>16} {'Epoch(h)':>10} {'Gen TPS':>12} {'Train TPS':>12} {'Bottleneck':<12}")
print("-" * 68)
for resp_len in [1024, 2048, 4096, 8192]:
    rl = base_rl.model_copy(update={"avg_response_len": resp_len, "max_response_len": resp_len * 2})
    r = perf.derive_targets(64, rl, gen_p, train_p, 24)
    print(f"{resp_len:>16} {r.epoch_time_hours:>10.2f} {r.gen_tps_target:>12,.0f} "
          f"{r.train_tps_target:>12,.0f} {r.bottleneck:<12}")

## Hardware Comparison: 910C vs CloudMatrix 384

In [ ]:
print(f"{'Hardware':<20} {'Peak TFLOPS':>12} {'HBM (GB)':>10} {'Epoch(h)':>10} {'Gen TPS':>12} {'Feasible':>8}")
print("-" * 80)
for hw_name, hw in [("Ascend 910C", hw_910c), ("CloudMatrix 384", hw_cm)]:
    p = LLMPerformanceModel(model, hw)
    r = p.derive_targets(64, base_rl, gen_p, train_p, 24)
    feasible = "OK" if (r.memory.train_feasible and r.memory.gen_feasible) else "OOM"
    print(f"{hw_name:<20} {hw.peak_tflops_bf16:>12.0f} {hw.hbm_capacity_gb:>10.0f} "
          f"{r.epoch_time_hours:>10.2f} {r.gen_tps_target:>12,.0f} {feasible:>8}")

## MoE: Finding the Right EP

In [ ]:
model_moe = load_model_config("../configs/models/qwen3_235b_moe.yaml")
perf_moe = LLMPerformanceModel(model_moe, hw_910c)

print(f"{'EP':>4} {'Epoch(h)':>10} {'Train Mem (GB)':>15} {'Gen Mem (GB)':>14} {'Train Feasible':>15} {'Gen Feasible':>13}")
print("-" * 78)
for ep in [1, 2, 4, 8, 16]:
    tp = 8
    # For MoE: EP must divide into total devices cleanly alongside TP
    # Use 128 total devices: TP=8, EP=ep, DP = 128 // (tp * ep)
    total = 128
    dp = max(1, total // (tp * ep))
    g_p = ParallelismConfig(tp=tp, dp=max(1, total // tp))
    t_p = ParallelismConfig(tp=tp, dp=dp, ep=ep)
    r = perf_moe.derive_targets(total, base_rl, g_p, t_p, 24)
    mem = r.memory
    print(f"{ep:>4} {r.epoch_time_hours:>10.2f} {mem.total_train_gb:>15.1f} {mem.total_gen_gb:>14.1f} "
          f"{'OK' if mem.train_feasible else 'OOM':>15} {'OK' if mem.gen_feasible else 'OOM':>13}")